# Laboratorio de Práctica: VGGNet y GoogLeNet

**Curso:** Redes Neuronales y Aprendizaje Profundo — UNI, FIIS, Unidad de Posgrado
**Clases:** 4 y 5 — VGGNet ([Simonyan & Zisserman, 2015](https://arxiv.org/abs/1409.1556)) y GoogLeNet ([Szegedy et al., 2015](https://arxiv.org/abs/1409.4842))

## Objetivos

Al terminar este laboratorio usted podrá:

1. Verificar experimentalmente el argumento de campo receptivo que justifica apilar convoluciones $3\times3$ en VGGNet.
2. Implementar en PyTorch una VGGNet configurable (configuraciones A, B, D, E del artículo) y **reproducir el conteo de parámetros de la Tabla 2** del paper.
3. Implementar el **módulo Inception**, tanto en su versión ingenua como con reducción de dimensión, y verificar el ahorro de cómputo.
4. Construir GoogLeNet completa siguiendo la Tabla 1 del artículo, con sus dos clasificadores auxiliares.
5. Comparar ambas arquitecturas de forma empírica: número de parámetros, tiempo de inferencia y desempeño en un conjunto de datos pequeño (CIFAR-10).

**Formato:** las celdas marcadas con `# TODO` son las que usted debe completar. El resto es código de apoyo ya funcional.
Ejecute las celdas en orden. Se recomienda usar un entorno con GPU (Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*).

---


In [ ]:
# Configuración inicial
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__} | torchvision: {torchvision.__version__}")


## Parte A — VGGNet

### A.1 Campo receptivo: 3 filtros $3\times3$ vs. 1 filtro $7\times7$

El artículo de Simonyan y Zisserman argumenta que apilar $n$ convoluciones $3\times3$ (stride 1, sin pooling entre ellas)
produce el mismo campo receptivo que una sola convolución de tamaño $(2n+1)\times(2n+1)$, con menos parámetros y más no linealidad.

La fórmula general para el campo receptivo de una pila de capas convolucionales es:

$$
r_0 = 1, \qquad r_k = r_{k-1} + (k_k - 1)\cdot \prod_{i<k} s_i
$$

donde $k_k$ es el tamaño de kernel de la capa $k$ y $s_i$ el stride de la capa $i$. Para $s_i=1$ en todas las capas, esto se simplifica a
$r_k = 1 + k\cdot(\text{tamaño de kernel} - 1)$ tras $k$ capas idénticas.

**Ejercicio A.1.** Complete la función `receptive_field` de modo que calcule el campo receptivo de una pila de $n$ capas
convolucionales de kernel `k` y stride 1, y verifique con ella las afirmaciones del artículo (Sección 2.3):
que 2 capas $3\times3$ equivalen a 1 capa $5\times5$, y que 3 capas $3\times3$ equivalen a 1 capa $7\times7$.


In [ ]:
def receptive_field(n_layers: int, kernel_size: int, stride: int = 1) -> int:
    """Campo receptivo de una pila de n_layers capas conv. idénticas (stride=1, sin pooling entre ellas)."""
    # TODO: complete la fórmula r = 1 + n_layers * (kernel_size - 1)  [válida para stride=1]
    r = None  # <-- reemplace esta línea
    return r


# Verificación (no modifique estas líneas; deben imprimir True)
print("2 capas 3x3  ==  1 capa 5x5 :", receptive_field(2, 3) == 5)
print("3 capas 3x3  ==  1 capa 7x7 :", receptive_field(3, 3) == 7)
print("1 capa 3x3   ==  1 capa 3x3 :", receptive_field(1, 3) == 3)


### A.2 Parámetros: 3×(3×3) vs. 1×(7×7)

**Ejercicio A.2.** Complete `params_stack_3x3` y `params_single_kxk` para reproducir el argumento del artículo (Sección 2.3):
para $C$ canales de entrada y salida en todas las capas, una pila de 3 convoluciones $3\times3$ usa $3(3^2C^2)=27C^2$ parámetros
(sin contar sesgos), mientras que una única convolución $7\times7$ usa $7^2C^2=49C^2$ — un 81% más.


In [ ]:
def params_stack_3x3(C: int) -> int:
    """Parámetros (sin sesgos) de 3 conv. 3x3 apiladas, C canales de entrada y salida en cada una."""
    # TODO: complete
    return None


def params_single_kxk(C: int, k: int) -> int:
    """Parámetros (sin sesgos) de una única convolución k x k, C canales de entrada y salida."""
    # TODO: complete
    return None


C = 256
p_stack = params_stack_3x3(C)
p_single = params_single_kxk(C, 7)
print(f"Pila de 3x(3x3): {p_stack:,} parámetros")
print(f"Una capa 7x7:    {p_single:,} parámetros")
print(f"Incremento relativo de la capa 7x7: {100*(p_single - p_stack)/p_stack:.1f}%  (el artículo reporta 81%)")


### A.3 Implementación de VGGNet (configuraciones A–E)

La Tabla 1 del artículo describe cinco configuraciones (A, A-LRN, B, C, D, E) que sólo difieren en profundidad y en el uso
ocasional de convoluciones $1\times1$ (configuración C). Usaremos la notación habitual de PyTorch/torchvision para describir
cada configuración como una lista: un entero es el número de canales de una conv. $3\times3$; `'M'` es max-pool $2\times2$/stride 2.

**Ejercicio A.3.** Complete las configuraciones que faltan (`'A'` y `'D'`) siguiendo el Cuadro 1 de la clase / Tabla 1 del artículo.
`'B'` y `'E'` ya están dadas como referencia.


In [ ]:
VGG_CFGS = {
    # TODO: complete la configuración A (11 capas con pesos: 8 conv. + 3 FC)
    'A': [],   # <-- reemplace: [64, 'M', 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M']

    'B': [64, 64, 'M', 128, 128, 'M', 256, 256, 'M', 512, 512, 'M', 512, 512, 'M'],

    # Configuración C: igual que D pero con conv1x1 en vez de la 3ra conv3x3 de cada bloque de 256/512/512
    'C': [64, 64, 'M', 128, 128, 'M', 256, 256, ('C', 256), 'M',
          512, 512, ('C', 512), 'M', 512, 512, ('C', 512), 'M'],

    # TODO: complete la configuración D (VGG-16: 13 conv. + 3 FC = 16 capas con pesos)
    'D': [],  # <-- reemplace siguiendo el Cuadro 1 (bloques de 2,2,3,3,3 conv3x3)

    'E': [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 256, 'M',
          512, 512, 512, 512, 'M', 512, 512, 512, 512, 'M'],
}


class VGG(nn.Module):
    """VGGNet configurable, siguiendo la Sección 2.1-2.2 del artículo."""

    def __init__(self, cfg_name: str, num_classes: int = 1000, batch_norm: bool = False):
        super().__init__()
        self.features = self._make_layers(VGG_CFGS[cfg_name], batch_norm)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(4096, 4096),        nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

    @staticmethod
    def _make_layers(cfg, batch_norm):
        layers = []
        in_channels = 3
        for v in cfg:
            if v == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            elif isinstance(v, tuple) and v[0] == 'C':
                out_channels = v[1]
                layers += [nn.Conv2d(in_channels, out_channels, kernel_size=1)]
                if batch_norm:
                    layers += [nn.BatchNorm2d(out_channels)]
                layers += [nn.ReLU(inplace=True)]
                in_channels = out_channels
            else:
                out_channels = v
                layers += [nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)]
                if batch_norm:
                    layers += [nn.BatchNorm2d(out_channels)]
                layers += [nn.ReLU(inplace=True)]
                in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


### A.4 Verificación contra la Tabla 2 del artículo

Cuente los parámetros de cada configuración y compárelos con los valores reportados por Simonyan y Zisserman
(Tabla 2): **A/A-LRN: 133M, B: 133M, C: 134M, D: 138M, E: 144M**. Pequeñas diferencias (&lt;1%) son normales por
detalles de implementación (por ejemplo, si se cuentan los sesgos).


In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


for cfg_name in ['A', 'B', 'C', 'D', 'E']:
    model = VGG(cfg_name, num_classes=1000)
    n_params = count_params(model)
    print(f"VGG-{cfg_name}: {n_params/1e6:6.1f} M parámetros")

del model


**Pregunta A.4.** Ejecute la celda anterior. Si `VGG_CFGS['A']` y `VGG_CFGS['D']` quedaron vacíos (TODO sin resolver),
el conteo de parámetros de esas dos configuraciones será incorrecto o la construcción del modelo fallará.
Una vez completados los TODO de la sección A.3, compare sus resultados con la Tabla 2 del artículo y comente
las diferencias (si las hay) en una celda de texto.


*(Responda aquí)*

### A.5 Inferencia con VGG-16 preentrenada

`torchvision` incluye pesos preentrenados en ImageNet para VGG-16. Los usaremos para clasificar una imagen y
medir el tiempo de un forward pass — cifras que compararemos más adelante con GoogLeNet.

> **Nota:** esta celda descarga pesos (~528 MB) la primera vez que se ejecuta; requiere conexión a internet.


In [ ]:
from torchvision.models import vgg16, VGG16_Weights

weights = VGG16_Weights.IMAGENET1K_V1
vgg16_pretrained = vgg16(weights=weights).to(device).eval()
preprocess = weights.transforms()
categories = weights.meta["categories"]

print(f"VGG-16 preentrenada: {count_params(vgg16_pretrained)/1e6:.1f} M parámetros")

# Imagen de prueba (cualquier URL de ejemplo; reemplace por una propia si lo desea)
import urllib.request
from PIL import Image

url = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"
urllib.request.urlretrieve(url, "sample.jpg")
img = Image.open("sample.jpg").convert("RGB")
x = preprocess(img).unsqueeze(0).to(device)

with torch.no_grad():
    t0 = time.time()
    logits = vgg16_pretrained(x)
    t_vgg = time.time() - t0

probs = F.softmax(logits, dim=1)[0]
top5 = torch.topk(probs, 5)
print(f"\nTiempo de inferencia (1 imagen, {device}): {t_vgg*1000:.1f} ms")
print("Top-5 predicciones:")
for score, idx in zip(top5.values, top5.indices):
    print(f"  {categories[idx]:<25s}  {score.item()*100:5.2f}%")


---
## Parte B — GoogLeNet / Inception

### B.1 El módulo Inception

**Ejercicio B.1.** Complete la clase `InceptionNaive`, que implementa la **versión ingenua** del módulo (Figura 2(a) del artículo:
cuatro ramas en paralelo —conv $1\times1$, conv $3\times3$, conv $5\times5$, max-pool $3\times3$— cuyas salidas se concatenan
en la dimensión de canales. Todas las ramas deben preservar la resolución espacial (use `padding` adecuado) para que la
concatenación sea válida.


In [ ]:
class InceptionNaive(nn.Module):
    """Módulo Inception ingenuo (Fig. 2a): sin convoluciones 1x1 de reducción."""

    def __init__(self, in_channels, ch1x1, ch3x3, ch5x5, pool_proj):
        super().__init__()
        # TODO: rama 1 - conv 1x1
        self.branch1 = None  # nn.Sequential(nn.Conv2d(in_channels, ch1x1, kernel_size=1), nn.ReLU(inplace=True))

        # TODO: rama 2 - conv 3x3 (padding=1 para preservar resolución)
        self.branch2 = None

        # TODO: rama 3 - conv 5x5 (padding=2 para preservar resolución)
        self.branch3 = None

        # TODO: rama 4 - max pool 3x3 stride 1 (padding=1) seguido de conv 1x1 de proyección
        self.branch4 = None

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


**Ejercicio B.2.** Ahora complete `InceptionReduced`, la versión **con reducción de dimensión** (Figura 2(b)):
agregue una convolución $1\times1$ *antes* de las ramas $3\times3$ y $5\times5$, y una convolución $1\times1$ de
proyección *después* del max-pooling. Los seis argumentos corresponden exactamente a las seis columnas de parámetros
de la Tabla 1 del artículo: `ch1x1`, `ch3x3reduce`, `ch3x3`, `ch5x5reduce`, `ch5x5`, `pool_proj`.


In [ ]:
class InceptionReduced(nn.Module):
    """Módulo Inception con reducción de dimensión (Fig. 2b) — el usado en GoogLeNet."""

    def __init__(self, in_channels, ch1x1, ch3x3reduce, ch3x3, ch5x5reduce, ch5x5, pool_proj):
        super().__init__()
        # Rama 1: conv 1x1
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch1x1, kernel_size=1), nn.ReLU(inplace=True)
        )

        # TODO: rama 2 = conv1x1 (reducción a ch3x3reduce) -> ReLU -> conv3x3 (padding=1, a ch3x3) -> ReLU
        self.branch2 = None

        # TODO: rama 3 = conv1x1 (reducción a ch5x5reduce) -> ReLU -> conv5x5 (padding=2, a ch5x5) -> ReLU
        self.branch3 = None

        # TODO: rama 4 = maxpool3x3(stride=1, padding=1) -> conv1x1 (proyección a pool_proj) -> ReLU
        self.branch4 = None

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


### B.3 Verificación contra la fila `inception (3a)` de la Tabla 1

La fila `inception (3a)` de la Tabla 1 del artículo tiene: entrada $28\times28\times192$, y columnas
`#1x1=64, #3x3reduce=96, #3x3=128, #5x5reduce=16, #5x5=32, pool_proj=32`, con **159K parámetros** reportados
(sin contar sesgos) y salida $28\times28\times256$ ($64+128+32+32=256$, correcto).

Verifique con su implementación:


In [ ]:
inc_3a = InceptionReduced(in_channels=192, ch1x1=64, ch3x3reduce=96, ch3x3=128,
                          ch5x5reduce=16, ch5x5=32, pool_proj=32)

n_params = count_params(inc_3a)
print(f"Parámetros de inception(3a) implementado: {n_params:,}  (paper reporta ~159,000)")

x = torch.randn(2, 192, 28, 28)
y = inc_3a(x)
print(f"Forma de salida: {tuple(y.shape)}  (esperado: (2, 256, 28, 28))")
assert y.shape == (2, 256, 28, 28), "La forma de salida no coincide -- revise el padding de cada rama."


**Pregunta B.3.** El paper reporta 159K parámetros para `inception(3a)`, pero `count_params` probablemente
le dé un número ligeramente distinto. Investigue por qué (piense en los sesgos de las convoluciones) y anote
su explicación aquí.


*(Responda aquí)*

### B.4 Comparación de cómputo: módulo ingenuo vs. con reducción

**Ejercicio B.4.** Use `InceptionNaive` con los mismos valores de `ch1x1=64, ch3x3=128, ch5x5=32, pool_proj=32`
(ignorando las reducciones) sobre la misma entrada que `inception(3a)` ($192$ canales de entrada) y compare el
número de parámetros con la versión reducida. ¿Coincide con la explicación del artículo sobre por qué la versión
ingenua es "prohibitivamente costosa" al escalar?


In [ ]:
# TODO: instancie InceptionNaive con los mismos canales de entrada/salida que inc_3a
inc_3a_naive = None  # InceptionNaive(in_channels=192, ch1x1=64, ch3x3=128, ch5x5=32, pool_proj=32)

# TODO: cuente los parámetros de inc_3a_naive y compárelos con los de inc_3a (ya calculados arriba)
n_params_naive = None

print(f"Ingenuo:        {n_params_naive:,} parámetros")
print(f"Con reducción:  {n_params:,} parámetros")
print(f"Ahorro:         {100*(1 - n_params/n_params_naive):.1f}%")


### B.5 GoogLeNet completa

Ahora ensamblamos la red completa siguiendo el Cuadro 3 de la clase (Tabla 1 del artículo): un *stem* convolucional
inicial, 9 módulos Inception organizados en 3 etapas (3a-3b, 4a-4e, 5a-5b) separadas por max-pooling, y dos
clasificadores auxiliares conectados a la salida de (4a) y (4d) — Sección 5 del artículo.

**Ejercicio B.5.** Complete la clase `AuxClassifier` siguiendo exactamente la estructura descrita en el artículo:
1. Average pooling $5\times5$, stride 3.
2. Convolución $1\times1$ con 128 filtros + ReLU.
3. Capa FC de 1024 unidades + ReLU.
4. Dropout con 70% de tasa.
5. Capa lineal a `num_classes`.


In [ ]:
class AuxClassifier(nn.Module):
    """Clasificador auxiliar (Sección 5 del artículo), conectado a (4a) o (4d)."""

    def __init__(self, in_channels, num_classes=1000):
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size=5, stride=3)
        # TODO: conv 1x1 con 128 filtros + ReLU
        self.conv = None
        # TODO: capa FC de 1024 unidades. OJO: la entrada depende de la resolución tras avgpool+conv;
        # para una entrada 14x14 (etapa 4a/4d), avgpool(5,3) produce una salida 4x4, y con 128 canales
        # eso son 128*4*4 = 2048 valores de entrada a la capa FC.
        self.fc1 = None
        self.dropout = nn.Dropout(0.7)
        self.fc2 = None  # TODO: capa lineal de 1024 -> num_classes

    def forward(self, x):
        x = self.pool(x)
        x = F.relu(self.conv(x), inplace=True)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x), inplace=True)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class GoogLeNet(nn.Module):
    """GoogLeNet siguiendo la Tabla 1 de Szegedy et al. (2015)."""

    def __init__(self, num_classes=1000, aux_logits=True):
        super().__init__()
        self.aux_logits = aux_logits

        # --- Stem (primeras capas, sin módulos Inception) ---
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3), nn.ReLU(inplace=True))
        self.maxpool1 = nn.MaxPool2d(3, stride=2, ceil_mode=True)
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=1), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(inplace=True))
        self.maxpool2 = nn.MaxPool2d(3, stride=2, ceil_mode=True)

        # --- Etapa 3 ---
        self.inception3a = InceptionReduced(192, 64, 96, 128, 16, 32, 32)     # -> 256
        self.inception3b = InceptionReduced(256, 128, 128, 192, 32, 96, 64)   # -> 480
        self.maxpool3 = nn.MaxPool2d(3, stride=2, ceil_mode=True)

        # --- Etapa 4 ---
        self.inception4a = InceptionReduced(480, 192, 96, 208, 16, 48, 64)    # -> 512
        self.inception4b = InceptionReduced(512, 160, 112, 224, 24, 64, 64)   # -> 512
        self.inception4c = InceptionReduced(512, 128, 128, 256, 24, 64, 64)   # -> 512
        self.inception4d = InceptionReduced(512, 112, 144, 288, 32, 64, 64)   # -> 528
        self.inception4e = InceptionReduced(528, 256, 160, 320, 32, 128, 128) # -> 832
        self.maxpool4 = nn.MaxPool2d(3, stride=2, ceil_mode=True)

        # --- Etapa 5 ---
        self.inception5a = InceptionReduced(832, 256, 160, 320, 32, 128, 128) # -> 832
        self.inception5b = InceptionReduced(832, 384, 192, 384, 48, 128, 128) # -> 1024

        # --- Clasificador final ---
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)

        # --- Clasificadores auxiliares ---
        if aux_logits:
            self.aux1 = AuxClassifier(512, num_classes)   # a la salida de (4a)
            self.aux2 = AuxClassifier(528, num_classes)   # a la salida de (4d)
        else:
            self.aux1 = None
            self.aux2 = None

    def forward(self, x):
        x = self.conv1(x); x = self.maxpool1(x)
        x = self.conv2(x); x = self.conv3(x); x = self.maxpool2(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)

        x = self.inception4a(x)
        aux1 = self.aux1(x) if (self.aux_logits and self.training) else None

        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        aux2 = self.aux2(x) if (self.aux_logits and self.training) else None

        x = self.inception4e(x)
        x = self.maxpool4(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)

        if self.aux_logits and self.training:
            return x, aux1, aux2
        return x


### B.6 Verificación de forma y conteo de parámetros

Verifique que un tensor de entrada $224\times224$ produce logits de forma `(batch, num_classes)`, y compare el
número total de parámetros con la cifra del artículo (**≈5–6 millones**, 12× menos que AlexNet).


In [ ]:
model = GoogLeNet(num_classes=1000, aux_logits=True)
model.eval()  # en modo eval, forward() sólo devuelve los logits principales (sin las ramas auxiliares)

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model(x)
print(f"Forma de salida (eval): {tuple(out.shape)}   (esperado: (2, 1000))")

n_params = count_params(model)
print(f"Parámetros totales de GoogLeNet (nuestra implementación): {n_params/1e6:.2f} M")
print("(el artículo reporta un total de la red de ~5-6M sin contar las ramas auxiliares completas;")
print(" con las dos ramas auxiliares el total sube algo más -- compare ambos números)")


**Pregunta B.6.** Compare el conteo de parámetros de su `GoogLeNet` (sin las ramas auxiliares, es decir,
`aux_logits=False`) contra `torchvision.models.googlenet(weights=None)`. ¿Qué tan cerca está su implementación?
Liste posibles causas de cualquier diferencia (por ejemplo: `BatchNorm` en la versión de torchvision, que el
artículo original no usa).


In [ ]:
from torchvision.models import googlenet

tv_model = googlenet(weights=None, aux_logits=True, init_weights=False)
print(f"GoogLeNet (torchvision): {count_params(tv_model)/1e6:.2f} M parámetros")

model_no_aux = GoogLeNet(num_classes=1000, aux_logits=False)
print(f"GoogLeNet (nuestra, sin aux): {count_params(model_no_aux)/1e6:.2f} M parámetros")


*(Responda aquí)*

---
## Parte C — Comparación empírica VGG-16 vs. GoogLeNet

### C.1 Parámetros y tiempo de inferencia

**Ejercicio C.1.** Complete la tabla comparativa ejecutando un forward pass de VGG-16 y de GoogLeNet preentrenada
(`torchvision.models.googlenet`) sobre el mismo lote de entrada, y registrando parámetros y tiempo.


In [ ]:
from torchvision.models import googlenet, GoogLeNet_Weights

gnet_pretrained = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1).to(device).eval()

batch = torch.randn(8, 3, 224, 224).to(device)

def timed_inference(model, x, n_repeats=10):
    with torch.no_grad():
        # calentamiento
        for _ in range(3):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_repeats):
            _ = model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        return (time.time() - t0) / n_repeats

# TODO: mida el tiempo promedio de inferencia de vgg16_pretrained y de gnet_pretrained sobre `batch`
t_vgg16 = None    # timed_inference(vgg16_pretrained, batch)
t_googlenet = None  # timed_inference(gnet_pretrained, batch)

print(f"{'Modelo':<12}{'Parámetros (M)':>16}{'Tiempo/lote (ms)':>20}")
print(f"{'VGG-16':<12}{count_params(vgg16_pretrained)/1e6:16.1f}{t_vgg16*1000:20.1f}")
print(f"{'GoogLeNet':<12}{count_params(gnet_pretrained)/1e6:16.1f}{t_googlenet*1000:20.1f}")


**Pregunta C.1.** ¿El modelo con menos parámetros (GoogLeNet) es necesariamente el más rápido en su hardware?
Discuta por qué el número de parámetros no siempre predice el tiempo de inferencia (piense en el número de
operaciones —columna "ops" de la Tabla 1— y en cuán paralelizables son las distintas capas).


*(Responda aquí)*

### C.2 Entrenamiento en CIFAR-10

Para una comparación de desempeño con recursos limitados de laboratorio, entrenaremos versiones **reducidas** de
ambas arquitecturas desde cero en CIFAR-10 ($32\times32$, 10 clases) durante pocas épocas. No es una réplica del
experimento de ImageNet del artículo (eso tomaría semanas de GPU, como reportan los propios autores), pero permite
comparar la dinámica de entrenamiento y el efecto del número de parámetros a una escala manejable en clase.


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

classes = ('avión','auto','ave','gato','venado','perro','rana','caballo','barco','camión')


**Ejercicio C.2.** Complete `VGGCifar`, una VGG "pequeña" (similar en espíritu a la configuración A, pero adaptada
a imágenes de $32\times32$: menos max-pools, para no reducir la resolución a menos de $1\times1$) y una versión reducida
de GoogLeNet con menos módulos Inception (`GoogLeNetCifarMini`). El objetivo no es reproducir la red completa del
artículo, sino comparar el **mismo estilo de bloque** (pila $3\times3$ vs.\ módulo Inception) bajo un presupuesto de
parámetros similar.


In [ ]:
class VGGCifar(nn.Module):
    """VGG pequeña para CIFAR-10 (bloques 3x3 apilados, como en el artículo, con menos capas/pools)."""

    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: construya self.features como una pila de bloques (conv3x3 -> ReLU) x2 + maxpool,
        # con canales 64 -> 128 -> 256, similar en espíritu a la Sección 2.1 del artículo.
        # Sugerencia: 3 bloques de max-pool bastan para 32x32 -> 4x4.
        self.features = nn.Sequential(
            # bloque 1: 3 -> 64 -> 64, maxpool
            # bloque 2: 64 -> 128 -> 128, maxpool
            # bloque 3: 128 -> 256 -> 256, maxpool
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


class GoogLeNetCifarMini(nn.Module):
    """Versión mini de GoogLeNet para CIFAR-10: stem ligero + 2 módulos Inception."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # TODO: agregue dos módulos InceptionReduced encadenados, con un maxpool entre ellos,
        # eligiendo canales que mantengan el total de parámetros comparable a VGGCifar.
        self.inception_a = None  # InceptionReduced(64, ...)
        self.pool = nn.MaxPool2d(2)
        self.inception_b = None  # InceptionReduced(<canales de salida de inception_a>, ...)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = None  # nn.Linear(<canales de salida de inception_b>, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.inception_a(x)
        x = self.pool(x)
        x = self.inception_b(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


In [ ]:
vgg_cifar = VGGCifar(num_classes=10).to(device)
gnet_cifar = GoogLeNetCifarMini(num_classes=10).to(device)

print(f"VGGCifar:          {count_params(vgg_cifar)/1e3:.1f} K parámetros")
print(f"GoogLeNetCifarMini:{count_params(gnet_cifar)/1e3:.1f} K parámetros")


### C.3 Entrenamiento y comparación

**Ejercicio C.3.** Complete el bucle de entrenamiento (celda siguiente) y ejecútelo para ambos modelos.
Registre la exactitud en el conjunto de prueba y el tiempo total de entrenamiento de cada uno.


In [ ]:
def train_model(model, trainloader, testloader, epochs=5, lr=0.01):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)  # como en el artículo de VGG

    history = {"train_loss": [], "test_acc": []}
    t_start = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # TODO: paso de entrenamiento estándar:
            # 1) poner los gradientes en cero
            # 2) forward
            # 3) calcular la pérdida con criterion
            # 4) backward
            # 5) optimizer.step()
            pass

            running_loss += loss.item() * inputs.size(0)

        train_loss = running_loss / len(trainloader.dataset)

        # Evaluación
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        test_acc = 100.0 * correct / total

        history["train_loss"].append(train_loss)
        history["test_acc"].append(test_acc)
        print(f"Época {epoch+1}/{epochs} | pérdida entren.: {train_loss:.3f} | exactitud prueba: {test_acc:.2f}%")

    elapsed = time.time() - t_start
    print(f"Tiempo total de entrenamiento: {elapsed:.1f} s")
    return history, elapsed


print("=== Entrenando VGGCifar ===")
hist_vgg, t_vgg_train = train_model(vgg_cifar, trainloader, testloader, epochs=5)

print("\n=== Entrenando GoogLeNetCifarMini ===")
hist_gnet, t_gnet_train = train_model(gnet_cifar, trainloader, testloader, epochs=5)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist_vgg["train_loss"], label="VGGCifar")
axes[0].plot(hist_gnet["train_loss"], label="GoogLeNetCifarMini")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Pérdida de entrenamiento"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_vgg["test_acc"], label="VGGCifar")
axes[1].plot(hist_gnet["test_acc"], label="GoogLeNetCifarMini")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Exactitud en prueba (%)"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'Modelo':<20}{'Parámetros (K)':>16}{'Tiempo entren. (s)':>20}{'Exactitud final (%)':>22}")
print(f"{'VGGCifar':<20}{count_params(vgg_cifar)/1e3:16.1f}{t_vgg_train:20.1f}{hist_vgg['test_acc'][-1]:22.2f}")
print(f"{'GoogLeNetCifarMini':<20}{count_params(gnet_cifar)/1e3:16.1f}{t_gnet_train:20.1f}{hist_gnet['test_acc'][-1]:22.2f}")


---
## Parte D — Preguntas de cierre (entregable)

Responda de forma breve (3-5 líneas cada una) en las celdas de texto provistas.

**D.1.** Con base en sus resultados de la Parte C, ¿qué arquitectura logró mejor exactitud por cada mil parámetros
en CIFAR-10? ¿Coincide esto con el mensaje del artículo de GoogLeNet sobre eficiencia?


*(Responda aquí)*

**D.2.** El artículo de VGG usa *scale jittering* (entrenar con $S$ aleatorio en $[256,512]$) como forma de
aumento de datos. ¿Qué transformación del `transform_train` de la Parte C cumple un rol similar (aunque más simple)
para CIFAR-10? Justifique.


*(Responda aquí)*

**D.3.** Explique con sus propias palabras por qué los clasificadores auxiliares de GoogLeNet se implementan
para actuar *solo quando* `self.training` es verdadero (vea el `forward` de la clase `GoogLeNet`). ¿Qué pasaría si
se usaran también en modo evaluación?


*(Responda aquí)*

**D.4 (opcional, reto).** Modifique `VGG_CFGS['C']` para construir la configuración `C` completa tal como
aparece en el Cuadro 1 de la clase, y verifique que su conteo de parámetros se acerca a 134M (Tabla 2 del artículo).


In [ ]:
# Espacio para el reto D.4


---
### Referencias

- Simonyan, K., & Zisserman, A. (2015). *Very Deep Convolutional Networks for Large-Scale Image Recognition*. ICLR. [arXiv:1409.1556](https://arxiv.org/abs/1409.1556)
- Szegedy, C., Liu, W., Jia, Y., et al. (2015). *Going Deeper with Convolutions*. CVPR. [arXiv:1409.4842](https://arxiv.org/abs/1409.4842)
